# Fine-Tuning QLoRA do Qwen3-1.7B no RunPod

Execute as células em ordem. O projeto e os resultados devem permanecer em `/workspace`.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'GPU CUDA não encontrada no Pod.'
props = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', props.name)
print('VRAM:', round(props.total_memory / 1024**3, 2), 'GB')

## Localizar ou extrair o projeto

In [ ]:
workspace = Path('/workspace')
candidates = list(workspace.rglob('configs/default.yaml'))
if not candidates:
    archives = sorted(workspace.glob('Celx*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
    assert archives, 'Envie o ZIP Celx para /workspace.'
    destination = workspace / 'legacy-doc-project'
    destination.mkdir(exist_ok=True)
    shutil.unpack_archive(str(archives[0]), str(destination))
    candidates = list(destination.rglob('configs/default.yaml'))
assert candidates, 'configs/default.yaml não foi encontrado.'
repo_dir = candidates[0].parents[1]
print('Projeto:', repo_dir)

## Instalar dependências do projeto

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-r', str(repo_dir / 'requirements.txt')], check=True)

## Validar o dataset SFT sem contaminar o benchmark

In [ ]:
sft_dir = repo_dir / 'dataset/processed/sft'
train_dir = sft_dir / 'train'
validation_dir = sft_dir / 'validation'
assert train_dir.exists(), f'Dataset de treino ausente: {train_dir}'
assert validation_dir.exists(), f'Dataset de validação ausente: {validation_dir}'
print('Dataset SFT validado:', sft_dir)

## Treinar

O comando usa QLoRA em 4 bits e grava checkpoints no volume persistente.

In [ ]:
output_dir = repo_dir / 'models/qwen3-legacy-doc-qlora'
command = [
    sys.executable, 'scripts/train_qlora.py',
    '--config', str(repo_dir / 'configs/default.yaml'),
    '--model', 'Qwen/Qwen3-1.7B',
    '--output-dir', str(output_dir),
]
subprocess.run(command, cwd=repo_dir, check=True)

## Conferir e preparar o backup

In [ ]:
assert (output_dir / 'adapter_config.json').exists(), 'Adapter não foi salvo corretamente.'
print('Arquivos gerados:')
for path in sorted(output_dir.iterdir()):
    print('-', path.name)
backup = shutil.make_archive(str(repo_dir / 'qwen3-legacy-doc-qlora'), 'zip', output_dir)
print('Backup:', backup)